# ST7 Project 2026

## Algorithm
1. Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [24]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import sys
import h5py
# from pysem import parse_sem3d_traces
from pathlib import Path

# sys.path.append(str(Path("pysem/src").resolve()))

path_to_src = str(Path("pysem/src").resolve())
print(f"Adding {path_to_src} to sys.path")
if path_to_src not in sys.path:
    sys.path.append(path_to_src)

from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from util_funct.sbatch_and_wait import sbatch_and_wait
from util_funct.compute_misfit import compute_misfit
from util_funct.write_backward_spec import write_backward_spec_from_template
from util_funct.write_misfit_files import write_time_reversed_residual_files
from util_funct.load_global_xyz_tuples import load_global_xyz_tuples
from gradient_search_direction.util_funct.update_parameters_file import update_parameters_file

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Adding /usr/users/cea_seism/benede_gio/CEISM-Project/pysem/src to sys.path


## 1. Paths and parameters

In [21]:
SEM3D_CONFIG_RES_FOLDER_PATH = "./sem3d_config_files"

FORWARD_PROBLEM_MESHER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "MESHER.sbatch")
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "SOLVER.sbatch")

TRACES_SIMULATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "traces")
TRACES_OSSERVATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "Uobs")

SEM3D_CONFIG_RES_FOLDER_PATH_ADJ = "./sem3d_config_files_adj"

ADJOINT_SOURCES_FOLDER_NAME = ""    #MUST BE short, otherwise sem3d will complain
ADJOINT_SOURCES_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, ADJOINT_SOURCES_FOLDER_NAME)

STATIONS_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "stations.txt")

BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "template/input_backward_template.spec")
ADJOINT_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "SOLVER_ADJOINT.sbatch")

In [22]:
GRADIENT_SEARCH_DIRECTION_FOLDER_PATH = "./gradient_search_direction"

GRADIENT_SEARCH_DIRECTION_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_direction.sbatch")
GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH = os.path.join(GRADIENT_SEARCH_DIRECTION_FOLDER_PATH, "gradient_search_parameters_file.json")

LBFGS_STATE_FOLDER_PATH = "state"
LBFGS_OUTPUT_FOLDER_PATH = "outputs"


In [4]:
N_ITER = 10 # to be modified
LBFGS_MEM = 5 # Numero di iterazioni da ricordare


## 2. Load observed data d_obs

In [ ]:
obs_stream = ParseSEM3DH5Traces(
    wkdir=TRACES_OSSERVATED_FOLDER_PATH,
    format='h5',
    names=['Uobs'],
    variables=['Displ'],
    components=['x', 'y', 'z']
)

obs_monitor = obs_stream['Uobs']

## 3. Initial material m_0

In [ ]:
! python3 ./pysem/src/pysem/generate_h5_materials.py @@prop "la" "mu" "ds" @@tag "linear_gradient" @@dir "z" @@xlim -2000 2000 @@ylim -2000 2000 @@zlim -2000 250 @@step 200 200 200 @@pfx 'example'
! mv example* ./sem3d_config_files/forward_problem_step1

## 4. Algorithm

In [ ]:
# sys.argv = ['parse_sem3d_snapshots.py', '@@wkd', './sem3d_config_files/res', '@@begin_time', '0', '@@end_time', '2']
from pysem.parse_sem3d_snapshots import compute_gradients_main

In [ ]:


N_ITER = 10 #to be modified

sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)
stations = np.loadtxt(STATIONS_FILE_PATH) #read_stations_pos(STATIONS_FILE_PATH)

for n in range(N_ITER):
    # ── STEP 1 ────────────────────────────────────────
    sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)
    


    # ── STEP 2: MISFIT ────────────────────────────────────────────────────
    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, obs_monitor)
      
    
    # ── STEP 3 ────────────────────────────────────────

    time_reversed_residual = residual[::-1, :, :]

    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                    t_sim, 
                                                    OUTPUT_DIR=ADJOINT_SOURCES_FOLDER_PATH)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH, 
                                      output_backward_spec_path = SEM3D_CONFIG_RES_FOLDER_PATH,
                                      adjoint_sources_folder_path = ADJOINT_SOURCES_FOLDER_PATH, 
                                      stations = stations, 
                                      file_names = file_names)

    sbatch_and_wait(ADJOINT_PROBLEM_SOLVER_SBATCH_PATH)

    # ── STEP 4-5: GRADIENT & SEARCH DIRECTIONS ─────────────────────────────────────────────────
        
    update_parameters_file(parameters_file_path=GRADIENT_SEARCH_DIRECTION_PARAMETERS_SBATCH_PATH, 
                           n_iter=n, 
                           LBFGS_MEM=LBFGS_MEM, 
                           LBFGS_STATE_FOLDER_PATH = LBFGS_STATE_FOLDER_PATH, 
                           LBFGS_OUTPUT_FOLDER_PATH = LBFGS_OUTPUT_FOLDER_PATH)

    sbatch_and_wait(GRADIENT_SEARCH_DIRECTION_SBATCH_PATH)
    
    # ── STEP 6-7 : LINE SEARCH & UPDATE ───────────────────
    
    def modify_h5_g(h5_path, vec_to_add, mapping_list):    
        with h5py.File(h5_path, "r+") as fmesh:
            mat = fmesh["samples"][...]
            
            xMinGlob = fmesh.attrs["xMinGlob"]
            xMaxGlob = fmesh.attrs["xMaxGlob"]
            n__elems = fmesh.attrs["xStep"]  # attention au nom, voir remarque plus bas
            xStep = np.array([
                (xMaxGlob[k] - xMinGlob[k]) / n__elems[k]
                for k in range(3)
            ])

            for coord, val in zip(mapping_list, vec_to_add):
                coord = np.array(coord)
                # Conversion coord physique → indice grille
                coord_int = ((coord - xMinGlob) / xStep).astype(int)
                # Option: sécuriser les indices (éviter out of bounds)
                coord_int = tuple(np.clip(coord_int, 0, np.array(mat.shape) - 1))
                mat[coord_int] += val
            del fmesh["samples"]
            fmesh.create_dataset('samples', data=mat)
        return None
    
    # Initialization of the Backtracking Line Search (BLS)
    
    ### RETRIEVE THE GRADIENTS grad_la and grad_mu, as well as dir_la and dir_mu from AllGather
    alpha_la=1, alpha_mu=1 , c1=1e-4 , xi=0.5
    J_thresh = J + c1(alpha_la*np.dot(dir_la,grad_la) + alpha_mu*np.dot(dir_mu,grad_mu))
    J_learn = np.inf
    
    dir_la, dir_mu = ... # AllGather ...
    vec_to_add_la, vec_to_add_mu = 2*alpha_la*dir_la , 2*alpha_mu*dir_mu
    materials_paths = {'La': ... , 'Mu' : ...} # Enter the correct paths where the h5 material files are stored
    mapping_list = load_global_xyz_tuples(LBFGS_OUTPUT_FOLDER_PATH)
    modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
    modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)

    ## Backtracking loop ##
    while J_learn >= J_thresh:
        vec_to_add_la -= alpha_la*dir_la
        vec_to_add_mu -= alpha_mu*dir_mu
        modify_h5_g(materials_paths['La'],vec_to_add_la,mapping_list)
        print("la.h5 modified",flush=True)
        modify_h5_g(materials_paths['Mu'],vec_to_add_mu,mapping_list)
        print("mu.h5 modified",flush=True)
        # Launch SEM3D ('/workdir/match/la.h5','/workdir/match/mu.h5' have just been updated
        # Call sbatch SOLVER, and then compute the misfit
        sbatch_and_wait("SOLVER.sbatch")
        J_learn = compute_misfit(TRACES_SIMULATED_FOLDER_PATH,obs_monitor)
        alpha_la *= xi
        alpha_mu *= xi
    